# Consolidacion de parquet por tipo

Este notebook une todos los anos por tipo desde `data/processed` y genera:

- `data/processed/fetal.parquet`
- `data/processed/nofetal.parquet`
- `data/processed/nac.parquet`


In [5]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
processed_dir = project_root / "data" / "processed"

type_config = {
    "fetal": {
        "input_dir": processed_dir / "fetales",
        "pattern": "fet*.parquet",
        "output_file": processed_dir / "fetal.parquet",
    },
    "nofetal": {
        "input_dir": processed_dir / "no_fetales",
        "pattern": "nofet*.parquet",
        "output_file": processed_dir / "nofetal.parquet",
    },
    "nac": {
        "input_dir": processed_dir / "nacimientos",
        "pattern": "nac*.parquet",
        "output_file": processed_dir / "nac.parquet",
    },
}

for type_name, cfg in type_config.items():
    parquet_files = sorted(cfg["input_dir"].glob(cfg["pattern"]))

    if not parquet_files:
        print(f"[WARN] No se encontraron archivos para {type_name} en {cfg['input_dir']}")
        continue

    dfs = [pd.read_parquet(file_path) for file_path in parquet_files]
    merged_df = pd.concat(dfs, ignore_index=True)

    merged_df.to_parquet(cfg["output_file"], index=False)
    print(f"[OK] {cfg['output_file'].name}: {len(merged_df):,} filas desde {len(parquet_files)} archivos")


[OK] fetal.parquet: 134,975 filas desde 5 archivos
[OK] nofetal.parquet: 1,476,123 filas desde 5 archivos
[OK] nac.parquet: 2,741,930 filas desde 5 archivos
